# 01 — Ingestion — Loan Prediction Based on Customer Behavior

**Stage 1 of the Tier A pipeline** (`DOCS/STRUCTURE.md`). Acquire the three delivered files, prove
what they are, and record what is wrong with them *before* anyone cleans or splits anything.

This notebook does five things:

1. Loads all three files **exactly as delivered** and preserves immutable copies with SHA-256 hashes.
2. Validates the schema against a written contract — structure, ranges, internal consistency,
   cardinality.
3. Runs a **four-check pre-modelling audit**, which is where this project's headline finding comes
   from. All four checks fail.
4. Diagnoses `Sample Prediction Dataset.csv`, because whether it is ground truth or a formatting
   template determines what may legally be reported later.
5. Records governance and schedules the fairness audit, since this is a model that would score
   people for credit.

> **Governing question for the whole project:** *This dataset reliably produces a ~0.90-accuracy,
> ~0.94-AUC default classifier — is that a model a lender could deploy, or an artifact of how the
> file was built?*

In [1]:
import hashlib, json, shutil, sys, warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)

# --- locate the project root without hardcoding an absolute path -------------------------------
ROOT = Path.cwd()
while not (ROOT / "archive" / "Training Data.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ARCHIVE = ROOT / "archive"
RAW, PROC, REPORTS = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "reports"
for p in (RAW, PROC, REPORTS):
    p.mkdir(parents=True, exist_ok=True)

SRC = {
    "train": ARCHIVE / "Training Data.csv",
    "test": ARCHIVE / "Test Data.csv",
    "submission_template": ARCHIVE / "Sample Prediction Dataset.csv",
}

print("python      :", sys.version.split()[0])
print("pandas      :", pd.__version__, "| numpy:", np.__version__)
print("project root:", ROOT.name)
for k, v in SRC.items():
    print(f"  {k:20s} {v.name:32s} exists: {v.exists()}")

python      : 3.14.6
pandas      : 3.0.3 | numpy: 2.5.1
project root: Loan Prediction Based on Customer Behavior
  train                Training Data.csv                exists: True
  test                 Test Data.csv                    exists: True
  submission_template  Sample Prediction Dataset.csv    exists: True


## 1.1 Load exactly as delivered

Three files, loaded with no type coercion, no renaming, no cleaning. The only thing normalised in
this cell is the **identifier column name** — train ships `Id`, test ships `ID`, the submission
template ships `id`. That single inconsistency is logged rather than silently absorbed, because a
file that cannot agree with itself on the spelling of its primary key is telling you something
about how it was assembled.

In [2]:
train = pd.read_csv(SRC["train"], encoding="utf-8")
test = pd.read_csv(SRC["test"], encoding="utf-8")
sample = pd.read_csv(SRC["submission_template"], encoding="utf-8")

print("as delivered:")
for name, df in [("train", train), ("test", test), ("submission_template", sample)]:
    print(f"  {name:20s} {df.shape[0]:>7,} rows x {df.shape[1]:>2} cols   "
          f"id column: {df.columns[0]!r}")

ID_RENAMES = {"train": {"Id": "ID"}, "test": {}, "submission_template": {"id": "ID", "risk_flag": "Risk_Flag"}}
train = train.rename(columns=ID_RENAMES["train"])
sample = sample.rename(columns=ID_RENAMES["submission_template"])

TARGET = "Risk_Flag"
IDENTIFIER = "ID"
FEATURES = [c for c in train.columns if c not in (IDENTIFIER, TARGET)]
NUMERIC = ["Income", "Age", "Experience", "CURRENT_JOB_YRS", "CURRENT_HOUSE_YRS"]
CATEGORICAL = ["Married/Single", "House_Ownership", "Car_Ownership", "Profession", "CITY", "STATE"]

print(f"\ntarget: {TARGET!r} | identifier: {IDENTIFIER!r} | {len(FEATURES)} feature columns")
print(f"  numeric     ({len(NUMERIC)}): {NUMERIC}")
print(f"  categorical ({len(CATEGORICAL)}): {CATEGORICAL}")
assert set(FEATURES) == set(NUMERIC) | set(CATEGORICAL), "feature inventory must be exhaustive"
print("\nfeature inventory is MECE over the 11 feature columns: OK")

as delivered:
  train                252,000 rows x 13 cols   id column: 'Id'
  test                  28,000 rows x 12 cols   id column: 'ID'
  submission_template   28,000 rows x  2 cols   id column: 'id'

target: 'Risk_Flag' | identifier: 'ID' | 11 feature columns
  numeric     (5): ['Income', 'Age', 'Experience', 'CURRENT_JOB_YRS', 'CURRENT_HOUSE_YRS']
  categorical (6): ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']

feature inventory is MECE over the 11 feature columns: OK


In [3]:
print("dtypes (train):")
for c in train.columns:
    print(f"   {c:20s} {str(train[c].dtype)}")

print("\nnull counts — all three files:")
for name, df in [("train", train), ("test", test), ("submission_template", sample)]:
    print(f"  {name:20s} total nulls: {df.isna().sum().sum()}")

print("\nhead of train:")
print(train.head(3).to_string())

dtypes (train):
   ID                   int64
   Income               int64
   Age                  int64
   Experience           int64
   Married/Single       str
   House_Ownership      str
   Car_Ownership        str
   Profession           str
   CITY                 str
   STATE                str
   CURRENT_JOB_YRS      int64
   CURRENT_HOUSE_YRS    int64
   Risk_Flag            int64

null counts — all three files:
  train                total nulls: 0
  test                 total nulls: 0
  submission_template  total nulls: 0

head of train:
   ID   Income  Age  Experience Married/Single House_Ownership Car_Ownership           Profession       CITY           STATE  CURRENT_JOB_YRS  CURRENT_HOUSE_YRS  Risk_Flag
0   1  1303834   23           3         single          rented            no  Mechanical_engineer       Rewa  Madhya_Pradesh                3                 13          0
1   2  7574516   40          10         single          rented            no   Software_Developer   

Zero nulls in all three files — the assessment brief (`DOCS/Data_Assessment_Loan_Risk.md` §2) says
so and it reproduces exactly. The brief also warns that this means the data was *pre-sanitised*,
not that it is analysis-ready. §1.5 shows how right that warning was.

## 1.2 Lineage — immutable copies and hashes

Provenance chain, documented explicitly because the replication finding in §1.5 only makes sense
against a synthetic-generation chain:

**Univ.AI hackathon (synthetic applicant generator) → Kaggle "Loan Prediction Based on Customer
Behavior" (Subham Jain) → this `archive/` folder.**

There is no bureau, no loan book and no origination system upstream of this file. Every structural
oddity found below is a property of a generator, not of a lending business.

In [4]:
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for blk in iter(lambda: fh.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()


PULLED_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
lineage = []
for name, path in SRC.items():
    dest = RAW / f"{name}_raw_immutable.csv"
    shutil.copy2(path, dest)
    df = {"train": train, "test": test, "submission_template": sample}[name]
    lineage.append({
        "artifact": name,
        "source_file": path.name,
        "immutable_copy": dest.name,
        "sha256": sha256(path),
        "bytes": path.stat().st_size,
        "rows": len(df),
        "cols": df.shape[1],
        "columns": "|".join(df.columns),
        "pulled_at_utc": PULLED_AT,
        "provenance": "Univ.AI hackathon (synthetic) -> Kaggle (Subham Jain) -> archive/",
    })

lineage = pd.DataFrame(lineage)
lineage.to_csv(RAW / "_lineage.csv", index=False)
print(lineage[["artifact", "source_file", "rows", "cols", "bytes", "sha256"]].to_string(index=False))
print(f"\nimmutable copies written to data/raw/ at {PULLED_AT}")

           artifact                   source_file   rows  cols    bytes                                                           sha256
              train             Training Data.csv 252000    13 20119553 8d44998e60cc8f37248dc259fb3f54555a0cb1cf57487fb4ac12a45d93f30ef7
               test                 Test Data.csv  28000    12  2155042 ff057c486d187b9ce0c77f6c05564d9ecb545d2206d2171b30504375ddd354da
submission_template Sample Prediction Dataset.csv  28000     2   240908 6ee11ef2161fe0c659f74f45d50c56ed6d7667d2ca0e03baa468a7bda62bc81c

immutable copies written to data/raw/ at 2026-08-01T05:46:47+00:00


## 1.3 Schema validation

One row per rule, pass/fail, written to `data/raw/_schema_validation.csv`. Rules are declared
first and evaluated second, so a passing rule is evidence rather than a description of whatever
the data happened to contain.

Watch rule **STR-04** in particular. It passes, and it is the single most misleading pass in this
project.

In [5]:
rules = []


def check(rule_id, description, passed, detail):
    rules.append({"rule_id": rule_id, "description": description,
                  "result": "PASS" if passed else "FAIL", "detail": detail})


# --- structure ---------------------------------------------------------------------------------
check("STR-01", "train ID is unique", train[IDENTIFIER].is_unique,
      f"{train[IDENTIFIER].nunique():,} unique of {len(train):,}; range {train[IDENTIFIER].min()}-{train[IDENTIFIER].max()}")
check("STR-02", "test ID is unique", test[IDENTIFIER].is_unique,
      f"{test[IDENTIFIER].nunique():,} unique of {len(test):,}; range {test[IDENTIFIER].min()}-{test[IDENTIFIER].max()}")
check("STR-03", "submission template IDs == test IDs, in order",
      (sample[IDENTIFIER].values == test[IDENTIFIER].values).all(),
      "element-wise identical")
check("STR-04", "no fully duplicated rows in train", not train.duplicated().any(),
      f"{train.duplicated().sum()} duplicated rows  <-- SUSPECT: see audit check A-01")
check("STR-05", f"{TARGET} is binary 0/1", set(train[TARGET].unique()) == {0, 1},
      f"values {sorted(train[TARGET].unique())}, positive rate {train[TARGET].mean():.4f}")
check("STR-06", "train and test share the same 11 feature columns",
      list(test.columns) == [IDENTIFIER] + FEATURES,
      f"test columns: {list(test.columns)}")
check("STR-07", "no nulls anywhere", (train.isna().sum().sum() + test.isna().sum().sum()) == 0,
      f"train {train.isna().sum().sum()}, test {test.isna().sum().sum()}")

# --- ranges ------------------------------------------------------------------------------------
RANGES = {"Age": (21, 79), "Experience": (0, 20), "CURRENT_JOB_YRS": (0, 14),
          "CURRENT_HOUSE_YRS": (10, 14)}
for col, (lo, hi) in RANGES.items():
    ok = train[col].between(lo, hi).all() and test[col].between(lo, hi).all()
    check(f"RNG-{col}", f"{col} within [{lo}, {hi}]", ok,
          f"train [{train[col].min()}, {train[col].max()}]  test [{test[col].min()}, {test[col].max()}]")
check("RNG-Income", "Income positive", (train.Income > 0).all() and (test.Income > 0).all(),
      f"train [{train.Income.min():,}, {train.Income.max():,}]")

# --- internal consistency ----------------------------------------------------------------------
v = (train.CURRENT_JOB_YRS > train.Experience).sum()
check("CON-01", "CURRENT_JOB_YRS <= Experience", v == 0, f"{v:,} violations")
v = (train.Age - train.Experience < 18).sum()
check("CON-02", "implied career start age (Age - Experience) >= 18", v == 0,
      f"{v:,} violations ({v/len(train):.2%}); minimum implied start age = {(train.Age - train.Experience).min()}")
v = (train.CURRENT_HOUSE_YRS > train.Age - 18).sum()
check("CON-03", "CURRENT_HOUSE_YRS <= Age - 18", v == 0, f"{v:,} violations")

# --- cardinality -------------------------------------------------------------------------------
for col, expected in [("Profession", 51), ("CITY", 317), ("STATE", 29)]:
    ok = train[col].nunique() == expected == test[col].nunique()
    check(f"CRD-{col}", f"{col} has {expected} categories in both files", ok,
          f"train {train[col].nunique()}, test {test[col].nunique()}  "
          f"<-- counts match; VOCABULARY does not (audit check A-03)")

schema = pd.DataFrame(rules)
schema.to_csv(RAW / "_schema_validation.csv", index=False)
print(schema.to_string(index=False))
print(f"\n{(schema.result == 'PASS').sum()} pass, {(schema.result == 'FAIL').sum()} fail")

              rule_id                                       description result                                                                                                                                                                           detail
               STR-01                                train ID is unique   PASS                                                                                                                                        252,000 unique of 252,000; range 1-252000
               STR-02                                 test ID is unique   PASS                                                                                                                                           28,000 unique of 28,000; range 1-28000
               STR-03     submission template IDs == test IDs, in order   PASS                                                                                                                                                          

### Two failures and one misleading pass

**CON-02 fails on 32,189 rows (12.77%)** — applicants whose work experience exceeds their adult
lifetime. The minimum implied career start age is **1**. The brief (§2) called this a "synthetic
smell"; it is now a counted defect. **CON-03** fails the same way for residence tenure.

**STR-04 passes**, and that pass is the reason this dataset's published results are wrong. A
whole-row duplicate test includes `ID`, which is unique by construction. The check therefore
cannot fail, no matter what the other eleven columns contain. §1.5 runs it correctly.

**CRD-*** all pass — 51 professions, 317 cities, 29 states in both files. Cardinality matching is
not vocabulary matching, and §1.5 check A-03 shows the difference.

## 1.4 Distributions as delivered

Recorded before any cleaning so the "synthetic smell" the brief flagged in §2 is on the record with
numbers attached. A Kolmogorov–Smirnov test against the uniform distribution is the cheapest way to
ask "was this sampled, or observed?"

In [6]:
from scipy import stats

print("numeric summary (train):")
print(train[NUMERIC].describe().T.to_string())

print("\nshape tests — is this a sample from a population, or a draw from a generator?")
print(f"{'column':22s} {'KS D vs uniform':>16s} {'p':>12s} {'skew':>9s} {'kurtosis':>10s}")
for c in NUMERIC:
    x = train[c].to_numpy(dtype=float)
    u = (x - x.min()) / (x.max() - x.min())
    ks = stats.kstest(u, "uniform")
    print(f"{c:22s} {ks.statistic:>16.4f} {ks.pvalue:>12.3g} {stats.skew(x):>+9.4f} {stats.kurtosis(x):>+10.4f}")

print("\ncorrelation among the five numeric features and the target:")
print(train[NUMERIC + [TARGET]].corr().round(4).to_string())

numeric summary (train):
                      count          mean           std      min        25%        50%        75%        max
Income             252000.0  4.997117e+06  2.878311e+06  10310.0  2503015.0  5000694.5  7477502.0  9999938.0
Age                252000.0  4.995407e+01  1.706385e+01     21.0       35.0       50.0       65.0       79.0
Experience         252000.0  1.008444e+01  6.002590e+00      0.0        5.0       10.0       15.0       20.0
CURRENT_JOB_YRS    252000.0  6.333877e+00  3.647053e+00      0.0        3.0        6.0        9.0       14.0
CURRENT_HOUSE_YRS  252000.0  1.199779e+01  1.399037e+00     10.0       11.0       12.0       13.0       14.0

shape tests — is this a sample from a population, or a draw from a generator?
column                  KS D vs uniform            p      skew   kurtosis


Income                           0.0052     2.32e-06   +0.0060    -1.1986


Age                              0.0203     1.92e-90   -0.0053    -1.2048
Experience                       0.0450            0   -0.0128    -1.1949
CURRENT_JOB_YRS                  0.1384            0   +0.2731    -0.7860
CURRENT_HOUSE_YRS                0.1945            0   +0.0068    -1.2749

correlation among the five numeric features and the target:
                   Income     Age  Experience  CURRENT_JOB_YRS  CURRENT_HOUSE_YRS  Risk_Flag
Income             1.0000 -0.0007      0.0064           0.0070            -0.0024    -0.0031
Age               -0.0007  1.0000     -0.0011           0.0022            -0.0201    -0.0218
Experience         0.0064 -0.0011      1.0000           0.6461             0.0193    -0.0345
CURRENT_JOB_YRS    0.0070  0.0022      0.6461           1.0000             0.0054    -0.0169
CURRENT_HOUSE_YRS -0.0024 -0.0201      0.0193           0.0054             1.0000    -0.0044
Risk_Flag         -0.0031 -0.0218     -0.0345          -0.0169            -0.0044    

**`Income`, `Age` and `Experience` are mutually uncorrelated** (|r| ≤ 0.007) and near-uniform
(KS D = 0.005 / 0.020 / 0.045, |skew| ≤ 0.013). They were sampled independently. Real income is
right-skewed and rises with age and experience; here it does neither.

Two consequences, both load-bearing later:

1. Any engineered ratio of these columns — the brief §7 proposes `Job_Stability_Ratio` and
   `Career_Start_Ratio` — is a ratio of independent uniforms, and carries no information that its
   parents lack. We will still build and test them (Stage 4), but the expectation is set here.
2. The only material correlation in the block is `Experience` ↔ `CURRENT_JOB_YRS` = **0.646**,
   which is a generator constraint (`CURRENT_JOB_YRS ≤ Experience`, rule CON-01, zero violations),
   not a behavioural fact.

`CURRENT_HOUSE_YRS` deserves its own line: it takes **five values, 10 to 14, at 19–21% each**. It
is not a residential-stability measure. The brief's proposed `House_Stability_Flag` (≥ median)
would be a coin flip by construction.

In [7]:
print("CURRENT_HOUSE_YRS — the whole distribution:")
print((train.CURRENT_HOUSE_YRS.value_counts(normalize=True).sort_index().rename("share")
       .to_frame().assign(rows=train.CURRENT_HOUSE_YRS.value_counts().sort_index())).round(4).to_string())

print(f"\ntarget balance: {train[TARGET].mean():.4f} positive "
      f"({train[TARGET].sum():,} of {len(train):,}) — majority class = {1 - train[TARGET].mean():.4f}")
print("\ncategorical inventory:")
for c in CATEGORICAL:
    top = train[c].value_counts()
    print(f"  {c:18s} {train[c].nunique():>4d} categories | most common: "
          f"{top.index[0]!r} ({top.iloc[0]:,}) | rarest: {top.index[-1]!r} ({top.iloc[-1]:,})")

CURRENT_HOUSE_YRS — the whole distribution:


                    share   rows
CURRENT_HOUSE_YRS               
10                 0.1932  48674
11                 0.2058  51873
12                 0.2055  51791
13                 0.2010  50659
14                 0.1945  49003

target balance: 0.1230 positive (30,996 of 252,000) — majority class = 0.8770

categorical inventory:
  Married/Single        2 categories | most common: 'single' (226,272) | rarest: 'married' (25,728)
  House_Ownership       3 categories | most common: 'rented' (231,898) | rarest: 'norent_noown' (7,184)
  Car_Ownership         2 categories | most common: 'no' (176,000) | rarest: 'yes' (76,000)
  Profession           51 categories | most common: 'Physician' (5,957) | rarest: 'Engineer' (4,048)
  CITY                317 categories | most common: 'Vijayanagaram' (1,259) | rarest: 'Karaikudi' (431)
  STATE                29 categories | most common: 'Uttar_Pradesh' (28,400) | rarest: 'Sikkim' (608)


## 1.5 The four-check pre-modelling audit

This is the section the project exists for. Four checks, each under ten lines, each of which
should be run on any tabular file before a single model is fitted. **All four fail on this one.**

| # | Check | Question it asks |
|---|---|---|
| A-01 | Duplicate rows **excluding identifier columns** | Is each row an independent observation? |
| A-02 | Score the identifier as a feature | Does row position leak the target? |
| A-03 | Category vocabulary, train vs test, per column | Will the encoder recognise what it is scored on? |
| A-04 | Holdout membership in train | Is the holdout actually held out? |

In [8]:
audit = []


def audit_row(check_id, name, passed, headline, detail=""):
    audit.append({"check_id": check_id, "check": name, "result": "PASS" if passed else "FAIL",
                  "headline": headline, "detail": detail})
    print(f"[{'PASS' if passed else 'FAIL'}] {check_id} — {name}\n      {headline}")
    if detail:
        print(f"      {detail}")

### A-01 — duplicate rows, excluding the identifier

`train.duplicated()` returned zero (rule STR-04) because `ID` is unique on all 252,000 rows. The
correct check excludes surrogate keys and asks whether the *observation* repeats.

In [9]:
dup_mask = train.duplicated(subset=FEATURES, keep=False)
profile_key = train[FEATURES].astype(str).agg("|".join, axis=1)
n_profiles = profile_key.nunique()
group_sizes = profile_key.value_counts()

audit_row("A-01", "duplicate rows excluding identifier", not dup_mask.any(),
          f"{dup_mask.sum():,} of {len(train):,} rows ({dup_mask.mean():.2%}) are duplicates of another row",
          f"252,000 rows resolve to {n_profiles:,} distinct applicant profiles "
          f"(ratio {len(train)/n_profiles:.3f})")

print("\ngroup-size distribution over the {:,} distinct profiles:".format(n_profiles))
gs = group_sizes.value_counts().sort_index()
print(pd.DataFrame({"copies_per_profile": gs.index, "n_profiles": gs.values,
                    "rows_covered": gs.index * gs.values}).to_string(index=False))
print(f"\n  {(group_sizes == 1).sum():,} profiles appear once  ({(group_sizes == 1).sum():,} rows)")
print(f"  {(group_sizes > 1).sum():,} profiles appear 27-40 times "
      f"({group_sizes[group_sizes > 1].sum():,} rows, {group_sizes[group_sizes > 1].sum()/len(train):.2%})")
print("  nothing appears between 2 and 26 times — a bimodal group-size distribution with an empty")
print("  middle is a generator signature, not a data-entry pattern.")

[FAIL] A-01 — duplicate rows excluding identifier
      215,993 of 252,000 rows (85.71%) are duplicates of another row
      252,000 rows resolve to 42,007 distinct applicant profiles (ratio 5.999)

group-size distribution over the 42,007 distinct profiles:
 copies_per_profile  n_profiles  rows_covered
                  1       36007         36007
                 27           1            27
                 29          10           290
                 30          19           570
                 31          57          1767
                 32         159          5088
                 33         342         11286
                 34         642         21828
                 35         995         34825
                 36        1216         43776
                 37        1223         45251
                 38         894         33972
                 39         367         14313
                 40          75          3000

  36,007 profiles appear once  (36,007 rows)
  6,00

**252,000 rows are 42,007 applicants.** Six thousand of them were replicated 27–40 times each.

This single fact sets the leakage rule for the entire project: **splits are by applicant profile,
never by row.** A random 5-fold split places copies of the same applicant in the training fold and
the validation fold, and the model scores itself on rows it has memorised. Notebook `05` measures
what that is worth: AUC **0.94 → 0.49**.

It gets worse. The replicated copies do not always agree on the label.

In [10]:
grp = train.assign(_k=profile_key).groupby("_k")[TARGET].agg(["size", "sum", "mean"])
rep = grp[grp["size"] > 1]
agreement = pd.Series({
    "unanimous 0": int((rep["mean"] == 0).sum()),
    "unanimous 1": int((rep["mean"] == 1).sum()),
    "MIXED (internally contradictory)": int(((rep["mean"] > 0) & (rep["mean"] < 1)).sum()),
})
mixed_rows = int(rep.loc[(rep["mean"] > 0) & (rep["mean"] < 1), "size"].sum())
print(f"label agreement across the {len(rep):,} replicated profiles:")
print(agreement.to_frame("profiles").to_string())
print(f"\nrows sitting inside an internally contradictory profile: {mixed_rows:,} "
      f"({mixed_rows/len(train):.2%})")
print(f"positive rate within mixed profiles: min {rep['mean'][(rep['mean']>0)&(rep['mean']<1)].min():.3f}, "
      f"median {rep['mean'][(rep['mean']>0)&(rep['mean']<1)].median():.3f}, "
      f"max {rep['mean'][(rep['mean']>0)&(rep['mean']<1)].max():.3f}")

maj = train.assign(_k=profile_key).groupby("_k")[TARGET].transform("mean")
from sklearn.metrics import roc_auc_score
print(f"\nceiling a pure memoriser reaches in-sample, predicting each profile's own majority label:")
print(f"   accuracy {(((maj >= .5).astype(int)) == train[TARGET]).mean():.4f}   "
      f"AUC {roc_auc_score(train[TARGET], maj):.4f}")
print("   — this is the number the leaderboard is actually chasing, and it is fiction.")

label agreement across the 6,000 replicated profiles:
                                  profiles
unanimous 0                           4817
unanimous 1                              0
MIXED (internally contradictory)      1183

rows sitting inside an internally contradictory profile: 42,547 (16.88%)
positive rate within mixed profiles: min 0.135, median 0.568, max 0.919



ceiling a pure memoriser reaches in-sample, predicting each profile's own majority label:
   accuracy 0.9355   AUC 0.9776
   — this is the number the leaderboard is actually chasing, and it is fiction.


**Not one replicated profile is unanimously a defaulter.** 4,817 are unanimously non-default, and
1,183 are internally contradictory — the same applicant, byte-for-byte identical on all eleven
attributes, labelled both 0 and 1. Labels were drawn per-row inside a pre-selected risky subset;
they were never attached to applicants.

### A-02 — score the identifier as a feature

`ID` is dropped from models by convention rather than by reasoning. The check asks what happens if
you don't.

In [11]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

y = train[TARGET].to_numpy()
skf = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
id_auc = cross_val_score(HistGradientBoostingClassifier(random_state=RANDOM_STATE),
                         train[[IDENTIFIER]], y, cv=skf, scoring="roc_auc").mean()
id_raw_auc = roc_auc_score(y, train[IDENTIFIER])

audit_row("A-02", "identifier scored as a feature", id_auc < 0.52,
          f"HistGB on {IDENTIFIER!r} ALONE reaches AUC {id_auc:.4f} (raw {IDENTIFIER} as a score: {id_raw_auc:.4f})",
          "chance is 0.500; notebook 05 shows all eleven real features together reach 0.500")

print("\nwhy — default rate by position in the file (20 equal blocks of 12,600 rows):")
blocks = pd.DataFrame({
    "block": np.arange(len(train)) // 12600,
    "y": y,
    "is_singleton": (profile_key.map(group_sizes) == 1).to_numpy(),
}).groupby("block").agg(rows=("y", "size"), default_rate=("y", "mean"),
                        singleton_share=("is_singleton", "mean"))
print(blocks.round(4).to_string())

[FAIL] A-02 — identifier scored as a feature
      HistGB on 'ID' ALONE reaches AUC 0.6739 (raw ID as a score: 0.5283)
      chance is 0.500; notebook 05 shows all eleven real features together reach 0.500

why — default rate by position in the file (20 equal blocks of 12,600 rows):
        rows  default_rate  singleton_share
block                                      
0      12600        0.1499           0.0003
1      12600        0.1364           0.0000
2      12600        0.1111           0.0000
3      12600        0.1444           0.0000
4      12600        0.1030           0.0000
5      12600        0.1533           0.0000
6      12600        0.1624           0.0000
7      12600        0.0710           0.0000
8      12600        0.0764           0.0000
9      12600        0.0747           0.0000
10     12600        0.0520           0.0000
11     12600        0.1567           0.0000
12     12600        0.0469           0.0000
13     12600        0.0766           0.0000
14     12600

**The file is sorted.** The last ~36,000 rows are entirely singleton applicants and default at
**~20%**; everything before them is the replicated block, defaulting at **~11%**. `ID` is a row
counter that encodes which block a row came from, which is why it out-predicts every real feature
in the file.

The two blocks are compared properly in notebook `02`; the short version is that they are
statistically indistinguishable on every numeric feature and differ only in label rate.

### A-03 — category vocabulary, train vs test

Cardinality matched in rule CRD-* (51 / 317 / 29 in both files). Cardinality is not vocabulary.

In [12]:
vocab = []
unseen_any = np.zeros(len(test), dtype=bool)
for c in CATEGORICAL:
    tr_v, te_v = set(train[c].unique()), set(test[c].unique())
    unseen = ~test[c].isin(tr_v)
    unseen_any |= unseen.to_numpy()
    vocab.append({"column": c, "train_categories": len(tr_v), "test_categories": len(te_v),
                  "test_only_categories": len(te_v - tr_v), "train_only_categories": len(tr_v - te_v),
                  "test_rows_unseen": int(unseen.sum()), "test_rows_unseen_pct": round(unseen.mean(), 4)})
vocab = pd.DataFrame(vocab)
print(vocab.to_string(index=False))

audit_row("A-03", "category vocabulary matches across train and test", vocab.test_only_categories.sum() == 0,
          f"{vocab.test_only_categories.sum()} test-only categories; "
          f"{unseen_any.sum():,} of {len(test):,} test rows ({unseen_any.mean():.2%}) carry at least one",
          "an encoder fitted on train maps these to 'unknown' without raising anything")

print("\nthe cause — train and test disagree on a separator character:")
for c in ["Profession", "STATE"]:
    tr_only = sorted(set(train[c].unique()) - set(test[c].unique()))[:4]
    te_only = sorted(set(test[c].unique()) - set(train[c].unique()))[:4]
    print(f"  {c}:")
    for a, b in zip(tr_only, te_only):
        print(f"      train {a!r:38s} test {b!r}")

print("\nunderscore usage:")
for c in CATEGORICAL:
    print(f"  {c:18s} train categories containing '_': "
          f"{sum('_' in v for v in train[c].unique()):>4d}/{train[c].nunique():<4d}   "
          f"test: {sum('_' in v for v in test[c].unique()):>4d}/{test[c].nunique()}")

         column  train_categories  test_categories  test_only_categories  train_only_categories  test_rows_unseen  test_rows_unseen_pct
 Married/Single                 2                2                     0                      0                 0                0.0000
House_Ownership                 3                3                     0                      0                 0                0.0000
  Car_Ownership                 2                2                     0                      0                 0                0.0000
     Profession                51               51                    23                     23             12848                0.4589
           CITY               317              317                    16                     16              1368                0.0489
          STATE                29               29                     8                      8             12503                0.4465
[FAIL] A-03 — category vocabulary matches across

  STATE:
      train 'Andhra_Pradesh'                       test 'Andhra Pradesh'
      train 'Himachal_Pradesh'                     test 'Himachal Pradesh'
      train 'Jammu_and_Kashmir'                    test 'Jammu and Kashmir'
      train 'Madhya_Pradesh'                       test 'Madhya Pradesh'

underscore usage:
  Married/Single     train categories containing '_':    0/2      test:    0/2
  House_Ownership    train categories containing '_':    1/3      test:    1/3
  Car_Ownership      train categories containing '_':    0/2      test:    0/2
  Profession         train categories containing '_':   23/51     test:    0/51
  CITY               train categories containing '_':   18/317    test:    2/317
  STATE              train categories containing '_':    8/29     test:    0/29


**Train writes `Software_Developer`; test writes `Software Developer`.** Twenty-three of fifty-one
professions and eight of twenty-nine states differ by a separator character alone, and that is
enough to make **72.45% of test rows** unrecognisable to any encoder fitted on the training file.

Notebook `02` reconciles this. It is the one genuine repair in the project, and it is what makes
the next check possible.

### A-04 — holdout membership in train

Run twice: as delivered, and after the §A-03 reconciliation. The difference between the two numbers
is the point.

In [13]:
def normalise_vocab(df):
    out = df.copy()
    for c in CATEGORICAL:
        out[c] = out[c].str.replace("_", " ", regex=False).str.strip()
    return out


k_train_raw = train[FEATURES].astype(str).agg("|".join, axis=1)
k_test_raw = test[FEATURES].astype(str).agg("|".join, axis=1)
k_train = normalise_vocab(train)[FEATURES].astype(str).agg("|".join, axis=1)
k_test = normalise_vocab(test)[FEATURES].astype(str).agg("|".join, axis=1)

match_raw = k_test_raw.isin(set(k_train_raw))
match_norm = k_test.isin(set(k_train))

audit_row("A-04", "holdout rows absent from train", not match_norm.any(),
          f"{match_norm.sum():,} of {len(test):,} test rows ({match_norm.mean():.2%}) have an exact "
          f"feature-vector twin in train",
          f"as delivered the same check finds only {match_raw.sum():,} ({match_raw.mean():.2%}) — "
          f"the vocabulary mismatch conceals the overlap")

lut = train.assign(_k=k_train).groupby("_k")[TARGET].mean()
hit = k_test.map(lut)
print(f"\nof the {int(hit.notna().sum()):,} matched test rows, the training label of their twin is:")
print(f"   unanimously 0 : {int((hit == 0).sum()):,}")
print(f"   unanimously 1 : {int((hit == 1).sum()):,}")
print(f"   contradictory : {int(((hit > 0) & (hit < 1)).sum()):,}")
print(f"\ntest set also self-replicates: {k_test.nunique():,} distinct feature vectors in "
      f"{len(test):,} rows (ratio {len(test)/k_test.nunique():.3f})")

[FAIL] A-04 — holdout rows absent from train
      24,003 of 28,000 test rows (85.72%) have an exact feature-vector twin in train
      as delivered the same check finds only 6,624 (23.66%) — the vocabulary mismatch conceals the overlap

of the 24,003 matched test rows, the training label of their twin is:
   unanimously 0 : 19,231
   unanimously 1 : 0
   contradictory : 4,772

test set also self-replicates: 9,922 distinct feature vectors in 28,000 rows (ratio 2.822)


In [14]:
audit = pd.DataFrame(audit)
audit.to_csv(PROC / "_audit_results.csv", index=False)
vocab.to_csv(PROC / "_vocabulary_check.csv", index=False)
print(audit[["check_id", "check", "result", "headline"]].to_string(index=False))
print(f"\n{(audit.result == 'FAIL').sum()} of {len(audit)} checks FAIL.")

check_id                                             check result                                                                        headline
    A-01               duplicate rows excluding identifier   FAIL                  215,993 of 252,000 rows (85.71%) are duplicates of another row
    A-02                    identifier scored as a feature   FAIL             HistGB on 'ID' ALONE reaches AUC 0.6739 (raw ID as a score: 0.5283)
    A-03 category vocabulary matches across train and test   FAIL 47 test-only categories; 20,287 of 28,000 test rows (72.45%) carry at least one
    A-04                    holdout rows absent from train   FAIL  24,003 of 28,000 test rows (85.72%) have an exact feature-vector twin in train

4 of 4 checks FAIL.


## 1.6 Is the submission template ground truth, or a formatting stub?

The brief (§6) proposes calibrating the decision threshold to the sample submission's ~12.8%
positive rate. That is only sound if the file carries information. It does not, and establishing
that **here** — before any modelling notebook runs — is what stops a fabricated test metric from
appearing later.

In [15]:
s = sample[TARGET].to_numpy()
merged = test.join(sample.set_index(IDENTIFIER), on=IDENTIFIER)

ser = pd.Series(s)
run_lengths = ser.groupby((ser != ser.shift()).cumsum()).agg(["size", "first"])
longest_run_of_1s = int(run_lengths.loc[run_lengths["first"] == 1, "size"].max())

print(f"positive rate            : {s.mean():.4f}   (train: {train[TARGET].mean():.4f})")
print(f"lag-1 autocorrelation    : {np.corrcoef(s[:-1], s[1:])[0, 1]:+.5f}")
print(f"longest run of 1s        : {longest_run_of_1s}   "
      f"(expected ~3-4 for an i.i.d. Bernoulli sequence at this rate)")
print("\nassociation with each test feature — if this file were a prediction, these would not be flat:")
for c in ["Married/Single", "House_Ownership", "Car_Ownership"]:
    rates = merged.groupby(c)[TARGET].mean().round(4).to_dict()
    print(f"  {c:18s} {rates}")
for c in NUMERIC:
    print(f"  AUC vs {c:20s} {roc_auc_score(merged[TARGET], merged[c]):.4f}")

positive rate            : 0.1283   (train: 0.1230)
lag-1 autocorrelation    : +0.07118
longest run of 1s        : 3   (expected ~3-4 for an i.i.d. Bernoulli sequence at this rate)

association with each test feature — if this file were a prediction, these would not be flat:
  Married/Single     {'married': 0.1322, 'single': 0.1279}
  House_Ownership    {'norent_noown': 0.1346, 'owned': 0.1497, 'rented': 0.127}
  Car_Ownership      {'no': 0.1256, 'yes': 0.1348}


  AUC vs Income               0.5042
  AUC vs Age                  0.5042
  AUC vs Experience           0.5091
  AUC vs CURRENT_JOB_YRS      0.5109
  AUC vs CURRENT_HOUSE_YRS    0.5028


**The submission template is random noise.** AUC 0.504 against every numeric feature, flat across
every categorical, near-zero serial correlation. Its 12.8% positive rate is a formatting
convenience, not a prior.

> **Constraint adopted for the whole project, from here on:** *no metric in this project is ever
> computed against `Test Data.csv`.* There is no ground truth for it. Any number that looked like
> test-set performance would be fabricated. Every performance figure reported in notebooks 04–06
> comes from cross-validation **inside the training file**.

## 1.7 Governance

The fairness question is scheduled rather than waved off, because this is a model that would score
people for credit on age, marital status and asset ownership.

In [16]:
governance = pd.DataFrame([
    {"item": "data subject", "value": "synthetic applicant records (generated), not real persons"},
    {"item": "direct identifiers", "value": "none — ID is a row counter, no name/DOB/account"},
    {"item": "special-category data", "value": "none present; Age and marital status are present and are protected-adjacent"},
    {"item": "licence", "value": "public Kaggle dataset, hackathon origin"},
    {"item": "safe to commit", "value": "yes — notebooks and report only; raw CSVs stay local per .gitignore"},
    {"item": "person-level fairness audit", "value": "APPLICABLE — credit scoring affects people; executed in notebook 05 §6.6"},
    {"item": "proxy-feature review", "value": "scheduled — CITY/STATE as regional proxies, notebook 05"},
    {"item": "retention", "value": "raw copies under data/raw/, excluded from version control"},
])
governance.to_csv(RAW / "_governance.csv", index=False)
print(governance.to_string(index=False))

                       item                                                                       value
               data subject                   synthetic applicant records (generated), not real persons
         direct identifiers                             none — ID is a row counter, no name/DOB/account
      special-category data none present; Age and marital status are present and are protected-adjacent
                    licence                                     public Kaggle dataset, hackathon origin
             safe to commit         yes — notebooks and report only; raw CSVs stay local per .gitignore
person-level fairness audit    APPLICABLE — credit scoring affects people; executed in notebook 05 §6.6
       proxy-feature review                     scheduled — CITY/STATE as regional proxies, notebook 05
                  retention                   raw copies under data/raw/, excluded from version control


In [17]:
key_figures = {
    "n_train_rows": int(len(train)),
    "n_test_rows": int(len(test)),
    "n_features": len(FEATURES),
    "target_positive_rate": float(train[TARGET].mean()),
    "majority_class_rate": float(1 - train[TARGET].mean()),
    "n_profiles": int(n_profiles),
    "n_duplicate_rows": int(dup_mask.sum()),
    "duplicate_row_share": float(dup_mask.mean()),
    "n_replicated_profiles": int((group_sizes > 1).sum()),
    "n_singleton_profiles": int((group_sizes == 1).sum()),
    "replicated_rows": int(group_sizes[group_sizes > 1].sum()),
    "min_copies": int(group_sizes[group_sizes > 1].min()),
    "max_copies": int(group_sizes.max()),
    "profiles_unanimous_0": int(agreement["unanimous 0"]),
    "profiles_unanimous_1": int(agreement["unanimous 1"]),
    "profiles_mixed": int(agreement["MIXED (internally contradictory)"]),
    "mixed_rows": mixed_rows,
    "memoriser_ceiling_accuracy": float((((maj >= .5).astype(int)) == train[TARGET]).mean()),
    "memoriser_ceiling_auc": float(roc_auc_score(train[TARGET], maj)),
    "id_alone_auc": float(id_auc),
    "id_raw_auc": float(id_raw_auc),
    "test_rows_unseen_category": int(unseen_any.sum()),
    "test_rows_unseen_category_pct": float(unseen_any.mean()),
    "test_overlap_raw": int(match_raw.sum()),
    "test_overlap_normalised": int(match_norm.sum()),
    "test_overlap_pct": float(match_norm.mean()),
    "test_distinct_vectors": int(k_test.nunique()),
    "career_start_violations": int((train.Age - train.Experience < 18).sum()),
    "min_implied_start_age": int((train.Age - train.Experience).min()),
    "sample_template_positive_rate": float(s.mean()),
    "sample_template_auc_vs_age": float(roc_auc_score(merged[TARGET], merged["Age"])),
}
(REPORTS / "_key_figures.json").write_text(json.dumps(key_figures, indent=2))
print(json.dumps(key_figures, indent=2))

{
  "n_train_rows": 252000,
  "n_test_rows": 28000,
  "n_features": 11,
  "target_positive_rate": 0.123,
  "majority_class_rate": 0.877,
  "n_profiles": 42007,
  "n_duplicate_rows": 215993,
  "duplicate_row_share": 0.8571150793650794,
  "n_replicated_profiles": 6000,
  "n_singleton_profiles": 36007,
  "replicated_rows": 215993,
  "min_copies": 27,
  "max_copies": 40,
  "profiles_unanimous_0": 4817,
  "profiles_unanimous_1": 0,
  "profiles_mixed": 1183,
  "mixed_rows": 42547,
  "memoriser_ceiling_accuracy": 0.9355277777777777,
  "memoriser_ceiling_auc": 0.9776297899989016,
  "id_alone_auc": 0.6738578690215595,
  "id_raw_auc": 0.5282608537003337,
  "test_rows_unseen_category": 20287,
  "test_rows_unseen_category_pct": 0.7245357142857143,
  "test_overlap_raw": 6624,
  "test_overlap_normalised": 24003,
  "test_overlap_pct": 0.85725,
  "test_distinct_vectors": 9922,
  "career_start_violations": 32189,
  "min_implied_start_age": 1,
  "sample_template_positive_rate": 0.12832142857142856,
  "s

## Stage 1 — Gate Checklist (`DOCS/STRUCTURE.md`)

- [x] **Raw data saved to `data/raw/`** (untouched copies of all three files, SHA-256 hashed)
- [x] **Source, timestamp, and row/column counts logged** — `data/raw/_lineage.csv`, with the
      provenance chain stated (Univ.AI generator → Kaggle → `archive/`)
- [x] **Schema matches expectations, or discrepancies are documented** — 17 rules in
      `_schema_validation.csv`; 2 fail (CON-02 career-start arithmetic, CON-03 residence tenure)
      and one pass (STR-04) is explicitly flagged as misleading
- [x] **PII / data governance review completed** — `data/raw/_governance.csv`; synthetic records,
      no direct identifiers, **person-level fairness audit marked APPLICABLE and scheduled**
- [x] **Pre-modelling audit run** (project-specific gate) — four checks, **four failures**,
      written to `data/processed/_audit_results.csv`
- [x] **Reporting constraint recorded** — the submission template is noise; no metric may be
      computed against `Test Data.csv`

### What notebook 02 must do with this

| Finding | Consequence for cleaning |
|---|---|
| 42,007 profiles, 6,000 replicated 27–40× | Build a `profile_id`; it becomes the unit of independence for every split and test |
| 1,183 contradictory profiles | Retain and quantify — they are the label-noise floor, not rows to drop |
| `ID` reaches AUC 0.674 | Drop from every feature set, after fitting it once as an exhibit |
| Two sorted blocks, 11% vs 20% | Derive `block` for diagnostics only; bar it from every model |
| 72% of test rows carry unseen categories | Reconcile vocabulary `_` → space, logged category by category |
| 85.72% test/train overlap | Persist a singleton-only variant so a clean split exists |